# Modified Bessel Functions — Colab Test Suite

Tests `modified_bessel_i_forward(x, nu)` and `modified_bessel_k_forward(x, nu)` against SciPy.

**No full PyTorch build.** Clones repo for source, compiles only the bessel
functions as a tiny C++/CUDA extension (~2 min), tests on CPU and GPU.

1. **Runtime > Change runtime type > T4 GPU > Save**
2. **Runtime > Run all**

In [ ]:
#@title Cell 1: Install deps
!pip install scipy ninja -q

In [ ]:
#@title Cell 2: Compile bessel extension (CPU + CUDA, ~1 min)
import shutil, os, glob

# Clear stale cache
cache_dir = os.path.expanduser('~/.cache/torch_extensions')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# Write standalone bessel header (no PyTorch build deps needed)
header = r"""
#pragma once
#include <cmath>
#include <cstdint>
#include <limits>

#ifdef __CUDACC__
#define BESSEL_HD __host__ __device__
#else
#define BESSEL_HD
#endif

template<typename T>
inline BESSEL_HD T modified_bessel_i0_forward(T x) {
    const T A[] = {
        -4.41534164647933937950e-18, +3.33079451882223809783e-17,
        -2.43127984654795469359e-16, +1.71539128555513303061e-15,
        -1.16853328779934516808e-14, +7.67618549860493561688e-14,
        -4.85644678311192946090e-13, +2.95505266312963983461e-12,
        -1.72682629144155570723e-11, +9.67580903537323691224e-11,
        -5.18979560163526290666e-10, +2.65982372468238665035e-09,
        -1.30002500998624804212e-08, +6.04699502254191894932e-08,
        -2.67079385394061173391e-07, +1.11738753912010371815e-06,
        -4.41673835845875056359e-06, +1.64484480707288970893e-05,
        -5.75419501008210370398e-05, +1.88502885095841655729e-04,
        -5.76375574538582365885e-04, +1.63947561694133579842e-03,
        -4.32430999505057594430e-03, +1.05464603945949983183e-02,
        -2.37374148058994688156e-02, +4.93052842396707084878e-02,
        -9.49010970480476444210e-02, +1.71620901522208775349e-01,
        -3.04682672343198398683e-01, +6.76795274409476084995e-01,
    };
    const T B[] = {
        -7.23318048787475395456e-18, -4.83050448594418207126e-18,
        +4.46562142029675999901e-17, +3.46122286769746109310e-17,
        -2.82762398051658348494e-16, -3.42548561967721913462e-16,
        +1.77256013305652638360e-15, +3.81168066935262242075e-15,
        -9.55484669882830764870e-15, -4.15056934728722208663e-14,
        +1.54008621752140982691e-14, +3.85277838274214270114e-13,
        +7.18012445138366623367e-13, -1.79417853150680611778e-12,
        -1.32158118404477131188e-11, -3.14991652796324136454e-11,
        +1.18891471078464383424e-11, +4.94060238822496958910e-10,
        +3.39623202570838634515e-09, +2.26666899049817806459e-08,
        +2.04891858946906374183e-07, +2.89137052083475648297e-06,
        +6.88975834691682398426e-05, +3.36911647825569408990e-03,
        +8.04490411014108831608e-01,
    };
    T p, q = 0.0;
    if (std::abs(x) <= T(8.0)) {
        T a = A[0];
        for (int i = 1; i < 30; i++) { p = q; q = a; a = ((std::abs(x)/T(2.0)) - T(2.0))*q - p + A[i]; }
        return std::exp(std::abs(x)) * (T(0.5) * (a - p));
    }
    T b = B[0];
    for (int i = 1; i < 25; i++) { p = q; q = b; b = (T(32.0)/std::abs(x) - T(2.0))*q - p + B[i]; }
    return std::exp(std::abs(x)) * (T(0.5) * (b - p)) / std::sqrt(std::abs(x));
}

template<typename T>
inline BESSEL_HD T modified_bessel_i1_forward(T x) {
    const T A[] = {
        +2.77791411276104639959e-18, -2.11142121435816608115e-17,
        +1.55363195773620046921e-16, -1.10559694773538630805e-15,
        +7.60068429473540693410e-15, -5.04218550472791168711e-14,
        +3.22379336594557470981e-13, -1.98397439776494371520e-12,
        +1.17361862988909016308e-11, -6.66348972350202774223e-11,
        +3.62559028155211703701e-10, -1.88724975172282928790e-09,
        +9.38153738649577178388e-09, -4.44505912879632808065e-08,
        +2.00329475355213526229e-07, -8.56872026469545474066e-07,
        +3.47025130813767847674e-06, -1.32731636560394358279e-05,
        +4.78156510755005422638e-05, -1.61760815825896745588e-04,
        +5.12285956168575772895e-04, -1.51357245063125314899e-03,
        +4.15642294431288815669e-03, -1.05640848946261981558e-02,
        +2.47264490306265168283e-02, -5.29459812080949914269e-02,
        +1.02643658689847095384e-01, -1.76416518357834055153e-01,
        +2.52587186443633654823e-01,
    };
    const T B[] = {
        +7.51729631084210481353e-18, +4.41434832307170791151e-18,
        -4.65030536848935832153e-17, -3.20952592199342395980e-17,
        +2.96262899764595013876e-16, +3.30820231092092828324e-16,
        -1.88035477551078244854e-15, -3.81440307243700780478e-15,
        +1.04202769841288027642e-14, +4.27244001671195135429e-14,
        -2.10154184277266431302e-14, -4.08355111109219731823e-13,
        -7.19855177624590851209e-13, +2.03562854414708950722e-12,
        +1.41258074366137813316e-11, +3.25260358301548823856e-11,
        -1.89749581235054123450e-11, -5.58974346219658380687e-10,
        -3.83538038596423702205e-09, -2.63146884688951950684e-08,
        -2.51223623787020892529e-07, -3.88256480887769039346e-06,
        -1.10588938762623716291e-04, -9.76109749136146840777e-03,
        +7.78576235018280120474e-01,
    };
    T p, q = 0.0;
    if (std::abs(x) <= T(8.0)) {
        T a = A[0];
        for (int i = 1; i < 29; i++) { p = q; q = a; a = ((std::abs(x)/T(2.0)) - T(2.0))*q - p + A[i]; }
        T r = T(0.5) * (a - p) * std::abs(x) * std::exp(std::abs(x));
        return x < T(0.0) ? -r : r;
    }
    T b = B[0];
    for (int i = 1; i < 25; i++) { p = q; q = b; b = (T(32.0)/std::abs(x) - T(2.0))*q - p + B[i]; }
    T r = std::exp(std::abs(x)) * (T(0.5) * (b - p)) / std::sqrt(std::abs(x));
    return x < T(0.0) ? -r : r;
}

template<typename T>
inline BESSEL_HD T modified_bessel_k0_forward(T x) {
    const T A[] = {
        +1.37446543561352307156e-16, +4.25981614279661018399e-14,
        +1.03496952576338420167e-11, +1.90451637722020886025e-09,
        +2.53479107902614945675e-07, +2.28621210311945178607e-05,
        +1.26461541144692592338e-03, +3.59799365153615016266e-02,
        +3.44289899924628486886e-01, -5.35327393233902768720e-01,
    };
    const T B[] = {
        +5.30043377268626276149e-18, -1.64758043015242134646e-17,
        +5.21039150503902756861e-17, -1.67823109680541210385e-16,
        +5.51205597852431940784e-16, -1.84859337734377901440e-15,
        +6.34007647740507060557e-15, -2.22751332699166985548e-14,
        +8.03289077536357521100e-14, -2.98009692317273043925e-13,
        +1.14034058820847496303e-12, -4.51459788337394416547e-12,
        +1.85594911495471785253e-11, -7.95748924447710747776e-11,
        +3.57739728140030116597e-10, -1.69753450938905987466e-09,
        +8.57403401741422608519e-09, -4.66048989768794782956e-08,
        +2.76681363944501510342e-07, -1.83175552271911948767e-06,
        +1.39498137188764993662e-05, -1.28495495816278026384e-04,
        +1.56988388573005337491e-03, -3.14481013119645005427e-02,
        +2.44030308206595545468e+00,
    };
    if (x == T(0.0)) return std::numeric_limits<T>::infinity();
    if (x < T(0.0)) return std::numeric_limits<T>::quiet_NaN();
    T p, q = 0.0;
    if (x <= T(2.0)) {
        T a = A[0];
        for (int i = 1; i < 10; i++) { p = q; q = a; a = (x*x - T(2.0))*q - p + A[i]; }
        return T(0.5)*(a - p) - std::log(T(0.5)*x)*modified_bessel_i0_forward(x);
    }
    T b = B[0];
    for (int i = 1; i < 25; i++) { p = q; q = b; b = (T(8.0)/x - T(2.0))*q - p + B[i]; }
    return std::exp(-x) * (T(0.5)*(b - p)) / std::sqrt(x);
}

template<typename T>
inline BESSEL_HD T modified_bessel_k1_forward(T x) {
    const T A[] = {
        -7.02386347938628759343e-18, -2.42744985051936593393e-15,
        -6.66690169419932900609e-13, -1.41148839263352776110e-10,
        -2.21338763073472585583e-08, -2.43340614156596823496e-06,
        -1.73028895751305206302e-04, -6.97572385963986435018e-03,
        -1.22611180822657148235e-01, -3.53155960776544875667e-01,
        +1.52530022733894777053e+00,
    };
    const T B[] = {
        -5.75674448366501715755e-18, +1.79405087314755922667e-17,
        -5.68946255844285935196e-17, +1.83809354436663880070e-16,
        -6.05704724837331885336e-16, +2.03870316562433424052e-15,
        -7.01983709041831346144e-15, +2.47715442448130437068e-14,
        -8.97670518232499435011e-14, +3.34841966607842919884e-13,
        -1.28917396095102890680e-12, +5.13963967348173025100e-12,
        -2.12996783842756842877e-11, +9.21831518760500529508e-11,
        -4.19035475934189648750e-10, +2.01504975519703286596e-09,
        -1.03457624656780970260e-08, +5.74108412545004946722e-08,
        -3.50196060308781257119e-07, +2.40648494783721712015e-06,
        -1.93619797416608296024e-05, +1.95215518471351631108e-04,
        -2.85781685962277938680e-03, +1.03923736576817238437e-01,
        +2.72062619048444266945e+00,
    };
    if (x == T(0.0)) return std::numeric_limits<T>::infinity();
    if (x < T(0.0)) return std::numeric_limits<T>::quiet_NaN();
    T p, q = 0.0;
    if (x <= T(2.0)) {
        T a = A[0];
        for (int i = 1; i < 11; i++) { p = q; q = a; a = (x*x - T(2.0))*q - p + A[i]; }
        return std::log(T(0.5)*x)*modified_bessel_i1_forward(x) + T(0.5)*(a - p)/x;
    }
    T b = B[0];
    for (int i = 1; i < 25; i++) { p = q; q = b; b = (T(8.0)/x - T(2.0))*q - p + B[i]; }
    return std::exp(-x) * (T(0.5)*(b - p)) / std::sqrt(x);
}

template<typename T> inline BESSEL_HD int bessel_gamma_sign(T x) {
    if (x >= T(0.0)) return 1;
    int64_t n = static_cast<int64_t>(std::floor(-x + T(1.0)));
    return (n % 2 == 0) ? 1 : -1;
}

template<typename T> inline BESSEL_HD T bessel_log_abs_gamma(T x) {
    if (x > T(0.0)) return std::lgamma(x);
    if (x == std::floor(x)) return std::numeric_limits<T>::infinity();
    const T pi = T(3.14159265358979323846);
    return std::log(pi) - std::log(std::abs(std::sin(pi*x))) - std::lgamma(T(1.0)-x);
}

template<typename T> inline BESSEL_HD T bessel_k_asymptotic(T x, T nu) {
    const T pi = T(3.14159265358979323846);
    T mu = T(4.0)*nu*nu, sum_val = T(1.0), term = T(1.0), prev = T(1.0);
    for (int k = 1; k < 30; k++) {
        term *= (mu - T((2*k-1)*(2*k-1)))/(T(8.0)*T(k)*x);
        T a = std::abs(term); if (a > prev || a < std::numeric_limits<T>::epsilon()*std::abs(sum_val)) break;
        sum_val += term; prev = a;
    }
    return std::sqrt(pi/(T(2.0)*x))*std::exp(-x)*sum_val;
}

template<typename T> inline BESSEL_HD void temme_ik(T mu, T x, T* K_mu, T* K_mu1) {
    const T pi = T(3.14159265358979323846);
    const T euler = T(0.5772156649015328606065120900824024310422);
    const T tol = std::numeric_limits<T>::epsilon()*T(10.0);
    T gp = std::tgamma(T(1.0)+mu)-T(1.0), gm = std::tgamma(T(1.0)-mu)-T(1.0);
    T a = std::log(x/T(2.0)), b = std::exp(mu*a), sigma = -a*mu;
    T c = std::abs(mu)<T(1e-10) ? T(1.0) : std::sin(pi*mu)/(mu*pi);
    T d; if (std::abs(sigma)<T(1e-5)){T s2=sigma*sigma; d=T(1.0)+s2/T(6.0)+s2*s2/T(120.0);} else d=std::sinh(sigma)/sigma;
    T g1,g2;
    if(std::abs(mu)<T(1e-10)){g1=-euler;g2=T(1.0);}
    else{T ip=T(1.0)/(T(1.0)+gp),im=T(1.0)/(T(1.0)+gm);g1=(im-ip)/(T(2.0)*mu);g2=(im+ip)/T(2.0);}
    T p=(T(1.0)+gp)/(T(2.0)*b),q=(T(1.0)+gm)*b/T(2.0);
    T f=(std::cosh(sigma)*g1+d*(-a)*g2)/c,h=p;
    T coef=T(1.0),sum=coef*f,sum1=coef*h,x2_4=x*x/T(4.0);
    for(int k=1;k<100;k++){
        T kT=T(k),den=kT*kT-mu*mu; if(std::abs(den)<T(1e-300))break;
        f=(kT*f+p+q)/den;p/=(kT-mu);q/=(kT+mu);h=p-kT*f;coef*=x2_4/kT;
        T ds=coef*f,ds1=coef*h;sum+=ds;sum1+=ds1;
        if(std::abs(ds)<tol*std::abs(sum)&&std::abs(ds1)<tol*std::abs(sum1))break;
    }
    *K_mu=sum;*K_mu1=T(2.0)*sum1/x;
}

template<typename T> inline BESSEL_HD void bessel_k_asymptotic_pair(T x, T mu, T* K_mu, T* K_mu1) {
    const T pi=T(3.14159265358979323846);
    T prefix=std::sqrt(pi/(T(2.0)*x))*std::exp(-x);
    T mu2=T(4.0)*mu*mu,sum=T(1.0),term=T(1.0),prev=T(1.0);
    for(int k=1;k<30;k++){term*=(mu2-T((2*k-1)*(2*k-1)))/(T(8.0)*T(k)*x);T a=std::abs(term);if(a>prev)break;if(a<std::numeric_limits<T>::epsilon()*std::abs(sum))break;sum+=term;prev=a;}
    *K_mu=prefix*sum;
    T mu1_2=T(4.0)*(mu+T(1.0))*(mu+T(1.0));T s1=T(1.0),t1=T(1.0),p1=T(1.0);
    for(int k=1;k<30;k++){t1*=(mu1_2-T((2*k-1)*(2*k-1)))/(T(8.0)*T(k)*x);T a=std::abs(t1);if(a>p1)break;if(a<std::numeric_limits<T>::epsilon()*std::abs(s1))break;s1+=t1;p1=a;}
    *K_mu1=prefix*s1;
}

template<typename T> inline BESSEL_HD T bessel_i_asymptotic(T x, T nu) {
    const T pi=T(3.14159265358979323846);
    T mu=T(4.0)*nu*nu,sum_val=T(1.0),term=T(1.0),prev=T(1.0);
    for(int k=1;k<30;k++){term*=-(mu-T((2*k-1)*(2*k-1)))/(T(8.0)*T(k)*x);T a=std::abs(term);if(a>prev||a<std::numeric_limits<T>::epsilon()*std::abs(sum_val))break;sum_val+=term;prev=a;}
    return std::exp(x)/std::sqrt(T(2.0)*pi*x)*sum_val;
}

template<typename T> inline BESSEL_HD T bessel_i_series(T x, T nu) {
    if(x==T(0.0)){if(std::abs(nu)<T(1e-10))return T(1.0);if(nu>T(0.0))return T(0.0);return std::numeric_limits<T>::infinity();}
    T half_x=x/T(2.0),ln_hx=std::log(half_x),result=T(0.0);
    for(int64_t k=0;k<300;k++){
        T ga=nu+T(k)+T(1.0),ln_num=(nu+T(2*k))*ln_hx,lkf=std::lgamma(T(k+1)),lg=bessel_log_abs_gamma(ga);
        int gs=bessel_gamma_sign(ga);T lat=ln_num-lkf-lg;if(lat<T(-700.0))break;
        T term=T(gs)*std::exp(lat);T pr=result;result+=term;
        if(k>10&&(result==pr||std::abs(term)<std::numeric_limits<T>::epsilon()*std::abs(result)))break;
    }
    return result;
}

template<typename T> inline BESSEL_HD T bessel_k_recurrence(T K_mu, T K_mu1, T mu, int64_t N, T x) {
    if(N==0)return K_mu;if(N==1)return K_mu1;
    T Kp=K_mu,Kc=K_mu1,ls=T(0.0);
    for(int64_t n=1;n<N;n++){T o=mu+T(n);T Kn=Kp+(T(2.0)*o/x)*Kc;Kp=Kc;Kc=Kn;
        T th=sizeof(T)>=8?T(1e300):T(1e30);if(std::abs(Kc)>th){T s=std::abs(Kc);Kp/=s;Kc/=s;ls+=std::log(s);}}
    if(ls>T(0.0)){if(ls>std::log(std::numeric_limits<T>::max()))return std::numeric_limits<T>::infinity();return Kc*std::exp(ls);}
    return Kc;
}

template<typename T> inline BESSEL_HD T modified_bessel_i_forward(T x, T nu) {
    if(x<T(0.0))return std::numeric_limits<T>::quiet_NaN();
    if(x==T(0.0)){if(std::abs(nu)<T(1e-10))return T(1.0);if(nu>T(0.0))return T(0.0);
        T na=std::abs(nu),nr=std::floor(na+T(0.5));if(std::abs(na-nr)<T(1e-10))return T(0.0);return std::numeric_limits<T>::infinity();}
    T na=std::abs(nu);
    if(na<T(1e-10))return modified_bessel_i0_forward(x);
    if(std::abs(na-T(1.0))<T(1e-10))return modified_bessel_i1_forward(x);
    if(nu<T(0.0)){T nr=std::floor(na+T(0.5));if(std::abs(na-nr)<T(1e-10))nu=na;}
    if(x>std::max(T(20.0),na*na/T(2.0))+na){
        T result=bessel_i_asymptotic(x,na);
        if(nu<T(0.0)){const T pi=T(3.14159265358979323846);T nf=std::floor(na+T(0.5));
            if(std::abs(na-nf)>T(1e-10)){result+=T(2.0)/pi*std::sin(na*pi)*bessel_k_asymptotic(x,na);}}
        return result;}
    return bessel_i_series(x,nu);
}

template<typename T> inline BESSEL_HD T modified_bessel_k_forward(T x, T nu) {
    if(x<=T(0.0)){if(x==T(0.0))return std::numeric_limits<T>::infinity();return std::numeric_limits<T>::quiet_NaN();}
    nu=std::abs(nu);
    if(nu<T(1e-10))return modified_bessel_k0_forward(x);
    if(std::abs(nu-T(1.0))<T(1e-10))return modified_bessel_k1_forward(x);
    if(x>std::max(T(20.0),nu*nu/T(2.0))+nu)return bessel_k_asymptotic(x,nu);
    int64_t N=static_cast<int64_t>(std::floor(nu+T(0.5)));T mu=nu-T(N);
    if(mu>T(0.5)){mu-=T(1.0);N+=1;}else if(mu<T(-0.5)){mu+=T(1.0);N-=1;}
    T Km,Km1;if(x<=T(2.0))temme_ik(mu,x,&Km,&Km1);else bessel_k_asymptotic_pair(x,mu,&Km,&Km1);
    return bessel_k_recurrence(Km,Km1,mu,N,x);
}
"""

with open('/tmp/bessel_functions.h', 'w') as f:
    f.write(header)
print(f'Wrote {len(header)} bytes to /tmp/bessel_functions.h')

import torch
from torch.utils.cpp_extension import load_inline

cpp_source = r"""
#include <torch/extension.h>
#include "bessel_functions.h"

torch::Tensor bessel_i_cpu(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes(), "x and nu must have same shape");
    auto out = torch::empty_like(x);
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_i_cpu", [&] {
        auto xp = x.data_ptr<scalar_t>(); auto np_ = nu.data_ptr<scalar_t>(); auto op = out.data_ptr<scalar_t>();
        for (int64_t i = 0; i < x.numel(); i++) op[i] = modified_bessel_i_forward(xp[i], np_[i]);
    });
    return out;
}

torch::Tensor bessel_k_cpu(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes(), "x and nu must have same shape");
    auto out = torch::empty_like(x);
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_k_cpu", [&] {
        auto xp = x.data_ptr<scalar_t>(); auto np_ = nu.data_ptr<scalar_t>(); auto op = out.data_ptr<scalar_t>();
        for (int64_t i = 0; i < x.numel(); i++) op[i] = modified_bessel_k_forward(xp[i], np_[i]);
    });
    return out;
}
"""

cuda_source = r"""
#include <torch/extension.h>
#include "bessel_functions.h"
#include <c10/cuda/CUDAException.h>

template<typename scalar_t> __global__ void bessel_i_kernel(const scalar_t* x, const scalar_t* nu, scalar_t* out, int64_t n) {
    int64_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) out[idx] = modified_bessel_i_forward(x[idx], nu[idx]);
}
template<typename scalar_t> __global__ void bessel_k_kernel(const scalar_t* x, const scalar_t* nu, scalar_t* out, int64_t n) {
    int64_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) out[idx] = modified_bessel_k_forward(x[idx], nu[idx]);
}

torch::Tensor bessel_i_cuda(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes());
    auto out = torch::empty_like(x); int64_t n = x.numel();
    int threads = 256, blocks = (n + threads - 1) / threads;
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_i_cuda", [&] {
        bessel_i_kernel<<<blocks, threads>>>(x.data_ptr<scalar_t>(), nu.data_ptr<scalar_t>(), out.data_ptr<scalar_t>(), n);
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    });
    return out;
}
torch::Tensor bessel_k_cuda(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes());
    auto out = torch::empty_like(x); int64_t n = x.numel();
    int threads = 256, blocks = (n + threads - 1) / threads;
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_k_cuda", [&] {
        bessel_k_kernel<<<blocks, threads>>>(x.data_ptr<scalar_t>(), nu.data_ptr<scalar_t>(), out.data_ptr<scalar_t>(), n);
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    });
    return out;
}
"""

build_dir = '/tmp/bessel_build'
os.makedirs(build_dir, exist_ok=True)

try:
    bessel_ext = load_inline(
        name='bessel_ext',
        cpp_sources=[cpp_source],
        cuda_sources=[cuda_source],
        extra_include_paths=['/tmp'],
        functions=['bessel_i_cpu', 'bessel_k_cpu', 'bessel_i_cuda', 'bessel_k_cuda'],
        build_directory=build_dir,
        verbose=True,
    )
except RuntimeError as e:
    print("BUILD FAILED. Compiler output:")
    print(str(e))
    # Also check for any log files
    for f in glob.glob(os.path.join(build_dir, '*.log')) + glob.glob(os.path.join(build_dir, '*.txt')):
        print(f"\n--- {f} ---")
        print(open(f).read()[-3000:])
    # Print generated source files for debugging
    for f in glob.glob(os.path.join(build_dir, '*.cpp')) + glob.glob(os.path.join(build_dir, '*.cu')):
        print(f"\n--- {f} (first 20 lines) ---")
        lines = open(f).readlines()[:20]
        print(''.join(lines))
    raise

def modified_bessel_i(x, nu):
    x, nu = torch.as_tensor(x), torch.as_tensor(nu)
    nu = nu.expand_as(x) if nu.dim() == 0 else nu
    x = x.expand_as(nu) if x.dim() == 0 else x
    if x.is_cuda: return bessel_ext.bessel_i_cuda(x.contiguous(), nu.contiguous())
    return bessel_ext.bessel_i_cpu(x.contiguous(), nu.contiguous())

def modified_bessel_k(x, nu):
    x, nu = torch.as_tensor(x), torch.as_tensor(nu)
    nu = nu.expand_as(x) if nu.dim() == 0 else nu
    x = x.expand_as(nu) if x.dim() == 0 else x
    if x.is_cuda: return bessel_ext.bessel_k_cuda(x.contiguous(), nu.contiguous())
    return bessel_ext.bessel_k_cpu(x.contiguous(), nu.contiguous())

# Smoke test
x = torch.tensor([1.0, 2.0, 3.0])
nu = torch.tensor([0.5, 1.0, 2.0])
print('I_nu(x):', modified_bessel_i(x, nu))
print('K_nu(x):', modified_bessel_k(x, nu))
print('\nExtension compiled OK!')

In [ ]:
#@title Cell 3: Test helpers
import scipy.special
import numpy as np
import time

def max_rel_error(ref, actual):
    ref = np.asarray(ref, dtype=np.float64)
    actual = np.asarray(actual, dtype=np.float64)
    mask = (ref != 0) & np.isfinite(ref) & np.isfinite(actual)
    if not mask.any(): return 0.0
    return float(np.max(np.abs((ref[mask] - actual[mask]) / ref[mask])))

class Results:
    def __init__(self):
        self.passed = self.failed = self.skipped = 0
        self.failures = []
    def ok(self, name, cond):
        if cond: self.passed += 1; print(f'  [PASS] {name}')
        else: self.failed += 1; self.failures.append(name); print(f'  [FAIL] {name}')
    def skip(self, name, r): self.skipped += 1; print(f'  [SKIP] {name} ({r})')
    def header(self, t): print(f"\n{'='*60}\n  {t}\n{'='*60}")
    def summary(self):
        t = self.passed + self.failed
        print(f"\n{'='*60}\n  RESULTS: {self.passed}/{t} passed, {self.failed} failed, {self.skipped} skipped\n{'='*60}")
        if self.failed == 0: print('  ALL TESTS PASSED')
        else:
            for f in self.failures: print(f'    - {f}')

R = Results()

In [ ]:
#@title Cell 4: Test 1 — Precision vs SciPy (CPU)
R.header('TEST 1: Precision vs SciPy (CPU)')

np.random.seed(42)
x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

for dtype, tol in [(torch.float32, 1e-3), (torch.float64, 1e-5)]:
    np_dtype = {torch.float32: np.float32, torch.float64: np.float64}[dtype]
    x_t = torch.tensor(x_np, dtype=dtype)
    nu_t = torch.tensor(nu_np, dtype=dtype)

    ref_i = scipy.special.iv(nu_np, x_np).astype(np_dtype)
    out_i = modified_bessel_i(x_t, nu_t).numpy()
    err_i = max_rel_error(ref_i, out_i)
    R.ok(f'CPU I_nu {dtype} max_rel_err={err_i:.2e} (tol={tol})', err_i < tol)

    ref_k = scipy.special.kv(nu_np, x_np).astype(np_dtype)
    out_k = modified_bessel_k(x_t, nu_t).numpy()
    err_k = max_rel_error(ref_k, out_k)
    R.ok(f'CPU K_nu {dtype} max_rel_err={err_k:.2e} (tol={tol})', err_k < tol)

In [ ]:
#@title Cell 5: Test 2 — Edge cases
R.header('TEST 2: Edge cases')

v = modified_bessel_i(torch.tensor([0.0]), torch.tensor([0.0])).item()
R.ok(f'I_0(0) = {v:.4f} (expect 1.0)', abs(v - 1.0) < 1e-6)

v = modified_bessel_i(torch.tensor([0.0]), torch.tensor([2.0])).item()
R.ok(f'I_2(0) = {v:.6f} (expect 0.0)', abs(v) < 1e-6)

v = modified_bessel_i(torch.tensor([0.0]), torch.tensor([0.5])).item()
R.ok(f'I_0.5(0) = {v:.6f} (expect 0.0)', abs(v) < 1e-6)

x_t = torch.tensor([1.0, 2.0, 5.0], dtype=torch.float64)
i_pos = modified_bessel_i(x_t, torch.tensor([2.0, 2.0, 2.0], dtype=torch.float64))
i_neg = modified_bessel_i(x_t, torch.tensor([-2.0, -2.0, -2.0], dtype=torch.float64))
err = max_rel_error(i_pos.numpy(), i_neg.numpy())
R.ok(f'I_{{-2}}(x) == I_2(x), rel_err={err:.2e}', err < 1e-10)

v = modified_bessel_i(torch.tensor([-1.0]), torch.tensor([1.0])).item()
R.ok('I_1(-1) = NaN', np.isnan(v))

v = modified_bessel_k(torch.tensor([0.0]), torch.tensor([1.0])).item()
R.ok(f'K_1(0) = inf', v == float('inf') or v > 1e30)

k_pos = modified_bessel_k(x_t, torch.tensor([2.5, 2.5, 2.5], dtype=torch.float64))
k_neg = modified_bessel_k(x_t, torch.tensor([-2.5, -2.5, -2.5], dtype=torch.float64))
err = max_rel_error(k_pos.numpy(), k_neg.numpy())
R.ok(f'K_{{-2.5}}(x) == K_2.5(x), rel_err={err:.2e}', err < 1e-10)

v = modified_bessel_k(torch.tensor([-1.0]), torch.tensor([1.0])).item()
R.ok('K_1(-1) = NaN', np.isnan(v))

In [ ]:
#@title Cell 6: Test 3 — Large + fractional orders
R.header('TEST 3: Large and fractional orders')

v = modified_bessel_i(torch.tensor([10.0], dtype=torch.float64), torch.tensor([50.0], dtype=torch.float64)).item()
ref = scipy.special.iv(50.0, 10.0)
err = abs(v - ref) / abs(ref) if ref != 0 else abs(v)
R.ok(f'I_50(10) rel_err={err:.2e}', err < 1e-3)

v = modified_bessel_k(torch.tensor([10.0], dtype=torch.float64), torch.tensor([50.0], dtype=torch.float64)).item()
ref = scipy.special.kv(50.0, 10.0)
err = abs(v - ref) / abs(ref) if ref != 0 else abs(v)
R.ok(f'K_50(10) rel_err={err:.2e}', err < 1e-3)

x_f = torch.tensor([1.0, 3.0, 7.0], dtype=torch.float64)
nu_f = torch.tensor([2.5, 2.5, 2.5], dtype=torch.float64)
ref_i = scipy.special.iv(2.5, x_f.numpy())
out_i = modified_bessel_i(x_f, nu_f).numpy()
R.ok(f'I_2.5(x) rel_err={max_rel_error(ref_i, out_i):.2e}', max_rel_error(ref_i, out_i) < 1e-5)

ref_k = scipy.special.kv(2.5, x_f.numpy())
out_k = modified_bessel_k(x_f, nu_f).numpy()
R.ok(f'K_2.5(x) rel_err={max_rel_error(ref_k, out_k):.2e}', max_rel_error(ref_k, out_k) < 1e-5)

In [ ]:
#@title Cell 7: Test 4 — Precision vs SciPy (CUDA)
R.header('TEST 4: Precision vs SciPy (CUDA)')

if not torch.cuda.is_available():
    R.skip('CUDA precision', 'No CUDA')
else:
    np.random.seed(123)
    x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
    nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

    for dtype, tol in [(torch.float32, 1e-3), (torch.float64, 1e-5)]:
        np_dtype = {torch.float32: np.float32, torch.float64: np.float64}[dtype]
        x_t = torch.tensor(x_np, dtype=dtype, device='cuda')
        nu_t = torch.tensor(nu_np, dtype=dtype, device='cuda')

        ref_i = scipy.special.iv(nu_np, x_np).astype(np_dtype)
        out_i = modified_bessel_i(x_t, nu_t).cpu().numpy()
        R.ok(f'CUDA I_nu {dtype} rel_err={max_rel_error(ref_i, out_i):.2e}', max_rel_error(ref_i, out_i) < tol)

        ref_k = scipy.special.kv(nu_np, x_np).astype(np_dtype)
        out_k = modified_bessel_k(x_t, nu_t).cpu().numpy()
        R.ok(f'CUDA K_nu {dtype} rel_err={max_rel_error(ref_k, out_k):.2e}', max_rel_error(ref_k, out_k) < tol)

In [ ]:
#@title Cell 8: Test 5 — CPU/CUDA parity
R.header('TEST 5: CPU/CUDA parity')

if not torch.cuda.is_available():
    R.skip('CPU/CUDA parity', 'No CUDA')
else:
    np.random.seed(456)
    x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
    nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)
    x_cpu = torch.tensor(x_np, dtype=torch.float64)
    nu_cpu = torch.tensor(nu_np, dtype=torch.float64)

    cpu_i = modified_bessel_i(x_cpu, nu_cpu)
    gpu_i = modified_bessel_i(x_cpu.cuda(), nu_cpu.cuda()).cpu()
    R.ok(f'I_nu parity rel_err={max_rel_error(cpu_i.numpy(), gpu_i.numpy()):.2e}', max_rel_error(cpu_i.numpy(), gpu_i.numpy()) < 1e-12)

    cpu_k = modified_bessel_k(x_cpu, nu_cpu)
    gpu_k = modified_bessel_k(x_cpu.cuda(), nu_cpu.cuda()).cpu()
    R.ok(f'K_nu parity rel_err={max_rel_error(cpu_k.numpy(), gpu_k.numpy()):.2e}', max_rel_error(cpu_k.numpy(), gpu_k.numpy()) < 1e-10)

    x32 = x_cpu.float(); nu32 = nu_cpu.float()
    cpu32 = modified_bessel_i(x32, nu32)
    gpu32 = modified_bessel_i(x32.cuda(), nu32.cuda()).cpu()
    R.ok(f'I_nu float32 parity rel_err={max_rel_error(cpu32.numpy(), gpu32.numpy()):.2e}', max_rel_error(cpu32.numpy(), gpu32.numpy()) < 1e-6)

In [ ]:
#@title Cell 9: Test 6 — CUDA stress test (1M elements)
R.header('TEST 6: CUDA stress test')

if not torch.cuda.is_available():
    R.skip('stress test', 'No CUDA')
else:
    N = 1_000_000
    x_big = torch.rand(N, dtype=torch.float32, device='cuda') * 20.0 + 0.1
    nu_big = torch.rand(N, dtype=torch.float32, device='cuda') * 20.0 - 10.0

    torch.cuda.synchronize(); t0 = time.perf_counter()
    out_i = modified_bessel_i(x_big, nu_big)
    torch.cuda.synchronize(); dt_i = time.perf_counter() - t0

    torch.cuda.synchronize(); t0 = time.perf_counter()
    out_k = modified_bessel_k(x_big, nu_big)
    torch.cuda.synchronize(); dt_k = time.perf_counter() - t0

    R.ok(f'I_nu 1M no crash, {dt_i:.3f}s', out_i.shape == (N,))
    R.ok(f'K_nu 1M no crash, {dt_k:.3f}s', out_k.shape == (N,))
    R.ok('I_nu no NaN', not torch.isnan(out_i).any().item())
    R.ok('K_nu no NaN (x>0.1)', not torch.isnan(out_k).any().item())
    R.ok('I_nu all finite', torch.isfinite(out_i).all().item())

In [ ]:
#@title Cell 10: Summary
R.summary()